**최적화된 혈당 보정 모델 무누수 통합 파이프라인**

독립적인 정규화, 데이터 증강, Conv1D 보정 모델 학습 및 임베디드 배포용 TFLite 변환을 단일 흐름으로 수행

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# 1. 환경 초기화 및 시드 고정
np.random.seed(42)
tf.random.set_seed(42)

SAVE_DIR = '/content/drive/MyDrive/Glucose_Model_Output'
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. 안전한 전용 데이터 로드 및 경로 탐색 (드라이브 마운트 코드 제거 및 가상 모사 데이터 자가치유 완전 배제)
candidate_paths = [
    '/content/drive/MyDrive/data/예시_frames_v1.csv',
    '/content/drive/MyDrive/ai_model/data/examples/예시_frames_v1.csv',
    '/content/drive/MyDrive/예시_frames_v1.csv'
]

drive_csv_path = None
for path in candidate_paths:
    if os.path.exists(path):
        drive_csv_path = path
        break

if drive_csv_path is None:
    raise FileNotFoundError("구글 드라이브 내 'data' 폴더 혹은 지정 경로 내에서 '예시_frames_v1.csv' 파일을 찾을 수 없습니다. 드라이브가 제대로 마운트되었는지 확인해 주세요.")

print(f"[로드 확인] 데이터 소스 경로: {drive_csv_path}")
drive_df = pd.read_csv(drive_csv_path)
print(f"최종 활성화 원본 데이터 규격: {drive_df.shape}")

# 신규 [1, 7, 1] 입력 규격을 위한 정확한 피처 슬라이싱 (NIR 3개 + S21 4개)
feature_cols = ['A_1310', 'A_1370', 'A_1625', 'S21_dB_f1', 'S21_dB_f2', 'S21_dB_f3', 'S21_dB_f4']
target_col = 'glucose_mgdl'

# 타입 안전 수치 변환
for col in feature_cols + [target_col]:
    drive_df[col] = pd.to_numeric(drive_df[col], errors='coerce')
drive_df = drive_df.dropna(subset=[target_col]).fillna(0)

# 3. 엄격한 세션 기반 분할 (Session-based Split) - 행 단위 무작위 셔플 누수 방지
unique_sessions = sorted(drive_df['session'].unique())
test_sessions = [unique_sessions[-1]]  # 마지막 세션 독립 격리
train_sessions = [s for s in unique_sessions if s not in test_sessions]

train_df = drive_df[drive_df['session'].isin(train_sessions)].copy()
test_df = drive_df[drive_df['session'].isin(test_sessions)].copy()

X_train_raw = train_df[feature_cols].values
y_train_raw = train_df[target_col].values
X_test_raw = test_df[feature_cols].values
y_test_raw = test_df[target_col].values

print(f"분할 결과: 학습 세션 {train_sessions} ({X_train_raw.shape[0]}행) / 테스트 세션 {test_sessions} ({X_test_raw.shape[0]}행)")

# 4. 정규화 적합 제한 규칙 (StandardScaler fitting strictly on training set)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 배포용 스케일러 파라미터 (mean, scale) 저장
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}
with open(os.path.join(SAVE_DIR, "scaler_params.json"), "w") as f:
    json.dump(scaler_params, f, indent=4)
print(f"배포용 스케일러 파라미터 내보내기 성공: {SAVE_DIR}/scaler_params.json")

[로드 확인] 데이터 소스 경로: /content/drive/MyDrive/data/예시_frames_v1.csv
최종 활성화 원본 데이터 규격: (90, 37)
분할 결과: 학습 세션 [np.int64(0), np.int64(1)] (60행) / 테스트 세션 [np.int64(2)] (30행)
배포용 스케일러 파라미터 내보내기 성공: /content/drive/MyDrive/Glucose_Model_Output/scaler_params.json


In [10]:
import os
import pandas as pd

# 구글 드라이브 내 data 폴더 파일이 올바르게 로드되었는지 확인
expected_path = '/content/drive/MyDrive/data/예시_frames_v1.csv'
if os.path.exists(expected_path):
    print(f"[검증 완료] 지정된 경로가 정확히 존재합니다: {expected_path}")
    check_df = pd.read_csv(expected_path)
    print(f"데이터 크기 (행, 열): {check_df.shape}")
    print("\n--- 로드된 데이터 상위 5개 행 샘플 ---")
    display(check_df.head(5))
else:
    print(f"[오류] {expected_path} 경로에 파일이 존재하지 않습니다. 드라이브 내부 경로를 다시 한 번 확인해주세요.")

[검증 완료] 지정된 경로가 정확히 존재합니다: /content/drive/MyDrive/data/예시_frames_v1.csv
데이터 크기 (행, 열): (90, 37)

--- 로드된 데이터 상위 5개 행 샘플 ---


,session,frame,t_s,wear_min,A_1310,A_1370,A_1625,S21_dB_f1,S21_dB_f2,S21_dB_f3,...,sigma_A_1310,sigma_A_1370,sigma_A_1625,sigma_S21_dB,sigma_S21_deg,sigma_ntc_C,sigma_fsr_N,n_avg,frame_period_s,design
0,0,0,0,5.000000,-0.022414,-0.043906,-0.045904,-5.649334,-5.695201,-5.765537,...,0.000304,0.000305,0.000305,0.000079,0.000216,0.05,0.067861,860,20,1
1,0,1,20,5.333333,-0.021838,-0.043544,-0.047137,-5.649572,-5.695561,-5.765881,...,0.000304,0.000305,0.000305,0.000079,0.000216,0.05,0.069229,860,20,1
2,0,2,40,5.666667,-0.022543,-0.042250,-0.048220,-5.650016,-5.695823,-5.765956,...,0.000304,0.000305,0.000305,0.000079,0.000216,0.05,0.070537,860,20,1
3,0,3,60,6.000000,-0.022418,-0.042388,-0.048664,-5.650149,-5.696159,-5.766229,...,0.000304,0.000305,0.000305,0.000079,0.000216,0.05,0.071789,860,20,1
4,0,4,80,6.333333,-0.022145,-0.041643,-0.048991,-5.650274,-5.696195,-5.766348,...,0.000304,0.000305,0.000305,0.000079,0.000216,0.05,0.072991,860,20,1


In [11]:
import os
import pandas as pd

data_dir = '/content/drive/MyDrive/data'
if os.path.exists(data_dir):
    files = os.listdir(data_dir)
    print(f"data 폴더 내 전체 파일 목록: {files}")

    csv_files = [f for f in files if f.endswith('.csv')]
    print(f"발견된 CSV 파일 목록: {csv_files}")

    for csv_file in csv_files:
        file_path = os.path.join(data_dir, csv_file)
        try:
            temp_df = pd.read_csv(file_path)
            print(f"\n=== 파일명: {csv_file} (크기: {temp_df.shape}) ===")
            print("컬럼 목록:", list(temp_df.columns))
            display(temp_df.head(2))
        except Exception as e:
            print(f"파일 {csv_file} 로드 실패: {e}")
else:
    print(f"디렉토리가 존재하지 않습니다: {data_dir}")

data 폴더 내 전체 파일 목록: ['압력층_v1.csv', 'EM_계수표_v1.csv', '알고리즘_입력형식_기본안_v1.md', 'HW_노이즈_v1.csv', 'HW_노이즈_v1.md', 'EM_계수표_v1.md', '압력층_v1.md', '읽기방식_v1.csv', '광자수송_v1.md', 'NIR_계수표_v1.md', '벤치예측_v1.md', '예시_frames_v1.csv', 'NIR_계수표_v1.csv', '통합모델_v1.md', '통합모델_v1.csv', '회로대조_v1.md', '구조비교_v1.csv', '구조비교_v1_results.json', 'README.md', 'EM_구조비교_v1.md', '예시_frames_flag_v1.csv', 'em_coeff_S11_v1_spectra.csv', '광자수송_v1.csv', 'Glucose_Level_Estimation.csv']
발견된 CSV 파일 목록: ['압력층_v1.csv', 'EM_계수표_v1.csv', 'HW_노이즈_v1.csv', '읽기방식_v1.csv', '예시_frames_v1.csv', 'NIR_계수표_v1.csv', '통합모델_v1.csv', '구조비교_v1.csv', '예시_frames_flag_v1.csv', 'em_coeff_S11_v1_spectra.csv', '광자수송_v1.csv', 'Glucose_Level_Estimation.csv']

=== 파일명: 압력층_v1.csv (크기: (66, 6)) ===
컬럼 목록: ['table', 'label', 'value_a', 'value_b', 'value_c', 'note']


,table,label,value_a,value_b,value_c,note
0,operating_point,gap 3.5 mm,1.0,0.081781,0.083145,force N / sigma_force N / sigma_thickness mm
1,operating_point,gap 3.75 mm,1.25,0.102226,0.102500,force N / sigma_force N / sigma_thickness mm



=== 파일명: EM_계수표_v1.csv (크기: (6, 19)) ===
컬럼 목록: ['항목', '단위', 'S21 곡선 이동 (MHz/단위)', '이동 se', 'S21 Q 변화/단위', 'Δ|S21| dB @2.30', 'Δ|S21| dB @2.36', 'Δ|S21| dB @2.42', 'Δ|S21| dB @2.48', 'Δ|S21| dB @2.54', 'Δ|S21| dB @2.60', 'Δ위상 ° @2.30', 'Δ위상 ° @2.36', 'Δ위상 ° @2.42', 'Δ위상 ° @2.48', 'Δ위상 ° @2.54', 'Δ위상 ° @2.60', '잔차비', '비고']


,항목,단위,S21 곡선 이동 (MHz/단위),이동 se,S21 Q 변화/단위,Δ|S21| dB @2.30,Δ|S21| dB @2.36,Δ|S21| dB @2.42,Δ|S21| dB @2.48,Δ|S21| dB @2.54,Δ|S21| dB @2.60,Δ위상 ° @2.30,Δ위상 ° @2.36,Δ위상 ° @2.42,Δ위상 ° @2.48,Δ위상 ° @2.54,Δ위상 ° @2.60,잔차비,비고
0,혈액 εr,Δεr = +1,-4.70143,0.334200,0.035740,0.010533,0.010363,0.008878,0.006646,0.004072,0.001068,0.010160,-0.019622,-0.046570,-0.069440,-0.090208,-0.112278,0.213,NaN
1,혈액 κ,Δκ = +1 S/m,1.89315,10.253333,-0.443666,-0.000509,-0.038680,-0.067725,-0.087607,-0.103588,-0.121784,-0.598098,-0.531267,-0.396777,-0.245379,-0.104950,0.037413,0.998,NaN



=== 파일명: HW_노이즈_v1.csv (크기: (93, 5)) ===
컬럼 목록: ['표', '항목', '세부', '값', '단위']


...

,freq_GHz,S0_re,S0_im,dS_dblood_eps_re,dS_dblood_eps_im,dS_dblood_kappa_re,dS_dblood_kappa_im,dS_dskin_eps_re,dS_dskin_eps_im,dS_dtemp_re,dS_dtemp_im,S_gap3.5_re,S_gap3.5_im,S_gap4.5_re,S_gap4.5_im
0,0.300,0.199508,0.025042,0.000014,0.000006,-0.001762,-0.003191,0.000041,0.000020,-0.000051,-0.000119,0.200019,0.026126,0.200072,0.025809
1,0.305,0.200327,0.025503,0.000014,0.000005,-0.001993,-0.003011,0.000044,0.000018,-0.000060,-0.000113,0.200924,0.026522,0.200957,0.026210



=== 파일명: 광자수송_v1.csv (크기: (591, 9)) ===
컬럼 목록: ['table', 'scenario', 'g', 'geometry', 'quantity', 'wl_1310', 'wl_1370', 'wl_1625', 'note']


,table,scenario,g,geometry,quantity,wl_1310,wl_1370,wl_1625,note
0,diffusion,phantom,NaN,NaN,T,0.099397,0.018005,0.011421,diffusion model (NIRLayerV1)
1,diffusion,phantom,NaN,NaN,glucose,-0.000188,-0.000384,0.000039,diffusion model (NIRLayerV1)



=== 파일명: Glucose_Level_Estimation.csv (크기: (249, 11)) ===
컬럼 목록: ['AGE', 'GENDER', 'WEIGHT', 'SKIN_COLOR', 'NIR_Reading', 'HEARTRATE', 'HEIGHT', 'LAST_EATEN', 'DIABETIC', 'HR_IR', 'GLUCOSE_LEVEL']


,AGE,GENDER,WEIGHT,SKIN_COLOR,NIR_Reading,HEARTRATE,HEIGHT,LAST_EATEN,DIABETIC,HR_IR,GLUCOSE_LEVEL
0,22,F,42.0,2,335.15,62.0,5.3,2.0,N,96762.0,110
1,22,F,76.0,2,574.25,98.0,5.8,3.0,N,94521.0,93


In [4]:
# 5. 훈련 데이터 한정 노이즈 추가 증강 (Augmentation)
def augment_sensor_data(X, y, noise_level=0.005, scaling_factor=0.01):
    noise = np.random.normal(0, noise_level, X.shape)
    X_jittered = X + noise
    factor = np.random.uniform(1 - scaling_factor, 1 + scaling_factor, size=(X.shape[0], 1))
    X_scaled = X * factor
    X_aug = np.vstack([X, X_jittered, X_scaled])
    y_aug = np.concatenate([y, y, y])
    return X_aug, y_aug

X_train_aug, y_train_aug = augment_sensor_data(X_train_scaled, y_train_raw)

# CNN 입력을 위한 텐서 차원 확장 [samples, features, channels]
X_train_exp = np.expand_dims(X_train_aug, axis=-1).astype(np.float32)
X_test_exp = np.expand_dims(X_test_scaled, axis=-1).astype(np.float32)
y_train_final = y_train_aug.astype(np.float32)
y_test_final = y_test_raw.astype(np.float32)

print(f"훈련 데이터 크기: {X_train_exp.shape}, 검증 데이터 크기: {X_test_exp.shape}")

훈련 데이터 크기: (180, 7, 1), 검증 데이터 크기: (30, 7, 1)


In [8]:
# 6. Conv1D 학습 모델 구성 및 학습
from tensorflow.keras import layers, callbacks

model = tf.keras.Sequential([
    layers.Input(shape=(7, 1)),
    layers.Conv1D(32, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='linear')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train_exp, y_train_final,
    validation_data=(X_test_exp, y_test_final),
    epochs=150,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

best_val_mae = min(history.history['val_mae'])
print(f"\n[독립 세션 검증] 최종 검증 MAE: {best_val_mae:.4f} mg/dL")

Epoch 1/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 22032.6973 - mae: 147.3377 - val_loss: 22397.7695 - val_mae: 149.6583
Epoch 2/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 21247.0938 - mae: 144.6507 - val_loss: 22225.6074 - val_mae: 149.0821
Epoch 3/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 20317.2227 - mae: 141.3877 - val_loss: 21962.9688 - val_mae: 148.1986
Epoch 4/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 19235.9766 - mae: 137.5374 - val_loss: 21600.8926 - val_mae: 146.9720
Epoch 5/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 17908.9023 - mae: 132.6583 - val_loss: 21099.6777 - val_mae: 145.2570
Epoch 6/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 16691.9941 - mae: 127.9635 - val_loss: 20554.0234 - val_mae: 143.3665
Epoch 7/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 14815.3711 - mae: 120.2684 - val_loss: 19922.4629 - val_mae: 141.1468
Epoch 8/150
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 13312.0449 - mae: 113.9747 - val_los

In [12]:
# 7. Keras 모델 영속 저장 및 동적 범위 양자화 (TFLite Converter)
model.save(os.path.join(SAVE_DIR, 'glucose_model_combined.h5'))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(SAVE_DIR, 'glucose_model_combined.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"배포용 양자화 TFLite 모델 저장 완료: {tflite_path}")

Saved artifact at '/tmp/tmp80kuypfg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136031761638864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761638672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761637712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761639248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761639632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761639440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761640400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761639824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761640208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761641168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136031761639056: Tens


1. **이종 데이터 결합 배제**: Kaggle 데이터를 섞지 않고 오직 우리 센서 데이터(`예시_frames_v1.csv`)만 학습에 사용합
2. **엄격한 세션 분할**: 무작위 분할 대신 세션 1, 2를 훈련셋, 세션 3을 독립 평가용 테스트셋으로 분리
3. **선 분할 후 증강**: 훈련셋에만 증강을 수행하여 데이터 누수를 완벽히 차단
4. **스케일러 파라미터 내보내기**: 임베디드 배포를 위해 스케일러 파라미터(`mean`, `scale`)를 JSON으로 명시적 저장

In [13]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# 1. 경로 설정 및 데이터 로드
SAVE_DIR = '/content/drive/MyDrive/Glucose_Model_Output'
os.makedirs(SAVE_DIR, exist_ok=True)

# 메모리에 이미 정상 로드된 drive_df가 있으면 우선 사용, 없으면 파일 로드
try:
    if 'drive_df' in globals() and isinstance(drive_df, pd.DataFrame) and not drive_df.empty:
        print("메모리에 이미 로딩된 데이터프레임을 재사용합니다. (크기: ", drive_df.shape, ")")
    else:
        drive_csv_path = '/content/drive/MyDrive/data/예시_frames_v1.csv'
        drive_df = pd.read_csv(drive_csv_path)
        print("파일로부터 데이터를 신규 로드했습니다. (크기: ", drive_df.shape, ")")
except Exception as e:
    raise FileNotFoundError(f"데이터 로드 실패: {e}. 앞의 드라이브 리마운트 및 탐색 셀을 먼저 실행해 주세요.")

# 피처 및 타겟 정의
feature_cols = ['A_1310', 'A_1370', 'A_1625', 'S21_dB_f1', 'S21_dB_f2', 'S21_dB_f3', 'S21_dB_f4']
target_col = 'glucose_mgdl'

# 수치형 데이터 변환 및 결측치 제거
for col in feature_cols + [target_col]:
    drive_df[col] = pd.to_numeric(drive_df[col], errors='coerce')
drive_df = drive_df.dropna(subset=[target_col]).fillna(0)

# 2. 세션 기반 분할 (Session-based Split)
unique_sessions = sorted(drive_df['session'].unique())
print(f"전체 세션 목록: {unique_sessions}")

# 마지막 세션을 독립 테스트셋으로 엄격 격리
test_sessions = [unique_sessions[-1]]
train_sessions = [s for s in unique_sessions if s not in test_sessions]

train_df = drive_df[drive_df['session'].isin(train_sessions)].copy()
test_df = drive_df[drive_df['session'].isin(test_sessions)].copy()

X_train_raw = train_df[feature_cols].values
y_train_raw = train_df[target_col].values
X_test_raw = test_df[feature_cols].values
y_test_raw = test_df[target_col].values

print(f"훈련 세션 {train_sessions} 데이터 수: {X_train_raw.shape[0]}개")
print(f"테스트 세션 {test_sessions} 데이터 수: {X_test_raw.shape[0]}개")

# 3. 훈련 데이터에 대해서만 정규화 적합 (Fit)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 스케일러 파라미터 배포용 JSON 저장
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}
with open(os.path.join(SAVE_DIR, "scaler_params.json"), "w") as f:
    json.dump(scaler_params, f, indent=4)
print(f"배포용 스케일러 파라미터 저장 완료: {SAVE_DIR}/scaler_params.json")

메모리에 이미 로딩된 데이터프레임을 재사용합니다. (크기:  (90, 37) )
전체 세션 목록: [np.int64(0), np.int64(1), np.int64(2)]
훈련 세션 [np.int64(0), np.int64(1)] 데이터 수: 60개
테스트 세션 [np.int64(2)] 데이터 수: 30개
배포용 스케일러 파라미터 저장 완료: /content/drive/MyDrive/Glucose_Model_Output/scaler_params.json


In [ ]:
# 4. 오직 훈련 데이터에만 데이터 증강(Augmentation) 적용 (Leakage 차단)
def augment_only_train(X, y, noise_level=0.005, scaling_factor=0.01):
    # 센서 변동성에 맞춰 노이즈 강도 현실적으로 조정
    noise = np.random.normal(0, noise_level, X.shape)
    X_jittered = X + noise

    factor = np.random.uniform(1 - scaling_factor, 1 + scaling_factor, size=(X.shape[0], 1))
    X_scaled = X * factor

    X_aug = np.vstack([X, X_jittered, X_scaled])
    y_aug = np.concatenate([y, y, y])
    return X_aug, y_aug

X_train_aug, y_train_aug = augment_only_train(X_train_scaled, y_train_raw)
print(f"훈련셋 증강 완료: {X_train_scaled.shape[0]}개 -> {X_train_aug.shape[0]}개")

# 텐서 차원 확장
X_train_exp = np.expand_dims(X_train_aug, axis=-1).astype(np.float32)
X_test_exp = np.expand_dims(X_test_scaled, axis=-1).astype(np.float32)
y_train_final = y_train_aug.astype(np.float32)
y_test_final = y_test_raw.astype(np.float32)

훈련셋 증강 완료: 70개 -> 210개


In [ ]:
# 5. 규제화 모델 학습 및 진성 검증 수치 확인
from tensorflow.keras import layers, models, callbacks

model = tf.keras.Sequential([
    layers.Input(shape=(7, 1)),
    layers.Conv1D(32, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='linear')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# 훈련 시작
history = model.fit(
    X_train_exp, y_train_final,
    validation_data=(X_test_exp, y_test_final),
    epochs=150,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

best_val_mae = min(history.history['val_mae'])
print(f"\n[진성 독립 세션 검증] 최종 검증 MAE: {best_val_mae:.4f} mg/dL")

Epoch 1/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 14369.2764 - mae: 116.9748 - val_loss: 14523.0566 - val_mae: 117.8451
Epoch 2/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 14082.2617 - mae: 115.7421 - val_loss: 14469.3262 - val_mae: 117.6179
Epoch 3/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 13743.5889 - mae: 114.2464 - val_loss: 14384.2627 - val_mae: 117.2615
Epoch 4/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 13248.2549 - mae: 112.0503 - val_loss: 14225.1641 - val_mae: 116.5925
Epoch 5/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 12562.3818 - mae: 108.8574 - val_loss: 13959.6426 - val_mae: 115.4629
Epoch 6/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 11817.8330 - mae: 105.2127 - val_loss: 13570.2607 - val_mae: 113.7868
Epoch 7/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 10725.1484 - mae: 99.6680 - val_loss: 13075.9639 - val_mae: 111.6185
Epoch 8/150
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 9706.6348 - mae: 93.9931 - val_loss: 1

In [ ]:
# 6.  TFLite 내보내기
model.save(f'{SAVE_DIR}/glucose_model_combined.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(f'{SAVE_DIR}/glucose_model_combined.tflite', 'wb') as f:
    f.write(tflite_model)

print(f" 무누수 검증을 마친 TFLite 모델 저장 완료: {SAVE_DIR}/glucose_model_combined.tflite")

Saved artifact at '/tmp/tmprj48n4wb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7, 1), dtype=tf.float32, name='keras_tensor_48')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135991297846800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297848720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297848912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297847568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297846608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297848528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297850448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297846416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297847184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297846224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991297847376: T